# Modelación en ingeniería
## Diseño de señales de entrada

*Profesor: David Ortiz-Puerta*

---

### Importancia del diseño de la entrada

La identificación de sistemas se basa en datos de entrada-salida. Por lo tanto, los datos disponibles determinan qué tan bueno puede llegar a ser el modelo identificado: ningún algoritmo, por sofisticado que sea, puede extraer información que la entrada no haya excitado en el sistema. Esto convierte al diseño de la señal de entrada en una decisión crítica del experimento, comparable en importancia a la elección de la estructura del modelo o del método de estimación.

La entrada cumple dos roles simultáneos. Primero, **fija el punto de operación**: en sistemas no lineales o linealizados, el régimen alrededor del cual se trabaja depende directamente del valor medio y la amplitud de $u(t)$. Segundo, **determina qué modos del sistema se excitan**: cada frecuencia presente en la entrada activa la dinámica del sistema en esa banda, mientras que las frecuencias ausentes permanecen invisibles para el identificador.

Los criterios de diseño relevantes son:

- **Contenido espectral ($\Phi_u(\omega)$):** define qué dinámicas serán observables. Una entrada con espectro estrecho solo permite identificar el sistema en esa banda.
- **Forma de la señal:** se cuantifica con el factor de cresta $C_r = \max|u(t)|^2 / \text{var}(u(t))$. Un factor de cresta bajo (mínimo 1, alcanzado por señales binarias) permite mayor potencia útil bajo una restricción de amplitud máxima.
- **Amplitud:** debe ser suficiente para que la respuesta supere el ruido, pero acotada para mantenerse dentro del rango de linealidad (si así se desea).
- **Duración del experimento:** típicamente entre 6 y 18 veces el tiempo de crecimiento del sistema, para capturar la dinámica completa.

Existe una tensión inherente entre **manipular el espectro** y **mantener un factor de cresta bajo**: las señales fáciles de moldear espectralmente (multisinusoides, ruido filtrado) tienden a tener factores de cresta altos, mientras que las señales binarias (factor de cresta = 1) son más rígidas espectralmente. Cada tipo de señal representa un compromiso distinto en este espacio.

En este notebook se implementan las señales de entrada más comunes en identificación: escalón, suma de sinusoides, PRBS y barrido en frecuencia (chirp).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from helpers import configure_plot_style, generate_input

def plot_input(t, u, title=''):
    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.plot(t, u, lw=1.2)
    ax.set_xlabel('t [s]')
    ax.set_ylabel('u(t)')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

configure_plot_style()

## Definición matemática de las señales

### Escalón

Cambio abrupto entre dos niveles en un instante $t_s$:

\begin{equation}
u(t) = \begin{cases} u_0, & t < t_s \\ u_1, & t \geq t_s \end{cases}
\end{equation}

Excita principalmente bajas frecuencias. Útil para obtener constantes de tiempo, ganancia estática y verificar linealidad.

### Suma de sinusoides (multisine)

Combinación lineal de $m$ componentes sinusoidales:

\begin{equation}
u(t) = \sum_{i=1}^{m} a_i \cos(\omega_i t + \varphi_i), \quad 0 \leq \omega_1 < \omega_2 < \dots < \omega_m
\end{equation}

Los grados de libertad son las amplitudes $\{a_i\}$, frecuencias $\{\omega_i\}$ y fases $\{\varphi_i\}$. Su espectro está concentrado en las frecuencias seleccionadas, lo que permite control fino del contenido espectral.

### PRBS (Pseudo-Random Binary Sequence)

Señal binaria periódica de período $M$, que alterna entre dos niveles $\{-a, +a\}$ según una secuencia generada por un registro de desplazamiento con realimentación lineal (LFSR). Su función de autocorrelación es:

\begin{equation}
R_u(\tau) = \begin{cases} a^2, & \tau = 0, \pm M, \pm 2M, \dots \\ -\dfrac{a^2}{M}, & \text{en otro caso} \end{cases}
\end{equation}

Su densidad espectral aproxima la de un ruido blanco para períodos grandes. Tiene factor de cresta igual a 1 (óptimo) y se parametriza por el período de reloj $T_s$, el período total $M = P/T_s$ y la amplitud $a$.

### Chirp (barrido en frecuencia)

Sinusoide cuya frecuencia varía linealmente entre $f_0$ y $f_1$ en un intervalo $T$:

\begin{equation}
u(t) = a \sin\left(2\pi \left(f_0 t + \frac{f_1 - f_0}{2T} t^2 \right)\right)
\end{equation}

Excita una banda continua de frecuencias en una sola corrida, lo que la hace eficiente para caracterización espectral.

In [ ]:
t = np.linspace(0, 20, 2000)

u_step      = generate_input(t, 'step', t_step=5, u0=0, u1=1)
u_multisine = generate_input(t, 'multisine', freqs=[0.1, 0.3, 0.7], amps=[1, 0.5, 0.3])
u_prbs      = generate_input(t, 'prbs', n_bits=8, t_bit=0.5, amplitude=1.0, seed=42)
u_chirp     = generate_input(t, 'chirp', f0=0.05, f1=1.0, amplitude=1.0)
u_none      = generate_input(t, 'none')

plot_input(t, u_step)